# [9665] Word2vec 2
Data file:
* sklearn : 20newsgroups

In [ ]:
from datetime import datetime
print(f'Run time: {datetime.now().strftime("%D %T")}')

Run time: 03/24/25 21:09:52


### Import libraries

In [ ]:
#%%time

#! pip install gensim

In [ ]:
import pandas as pd
import numpy as np
from sklearn.datasets import fetch_20newsgroups
from functools import lru_cache
import nltk
from nltk.corpus import wordnet
from nltk import pos_tag
from nltk.stem import WordNetLemmatizer
from gensim.utils import simple_preprocess
from gensim.parsing.preprocessing import STOPWORDS
from gensim.models.phrases import Phrases
from gensim.models.phrases import Phraser
from gensim.models import Word2Vec

In [ ]:
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger_eng')

[nltk_data] Downloading package stopwords to /Users/vj/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /Users/vj/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /Users/vj/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.


True

### Load data

In [ ]:
%%time

newsgroups = fetch_20newsgroups(subset='all',
                                shuffle=True,
                                random_state=42)

CPU times: user 250 ms, sys: 63.3 ms, total: 314 ms
Wall time: 320 ms


In [ ]:
type(newsgroups)

sklearn.utils._bunch.Bunch

In [ ]:
# Create a DataFrame
df = pd.DataFrame({
    'text': newsgroups.data,
    'target': newsgroups.target,
    'target_names': [newsgroups.target_names[i] for i in newsgroups.target]
})

In [ ]:
df.shape

(18846, 3)

### Examine data

In [ ]:
pd.set_option('max_colwidth', None)

In [ ]:
df.sample(3)

,text,target,target_names
10220,"From: lbutler@hubcap.clemson.edu (L Clator Butler Jr)\nSubject: Re: DID HE REALLY RISE???\nOrganization: Clemson University\nLines: 11\n\nmcovingt@aisun3.ai.uga.edu (Michael Covington) writes:\n>(2) Nobody ever displayed the dead body of Jesus, even though both the\n>Jewish and the Roman authorities would have gained a lot by doing so\n>(it would have discredited the Christians).\n\nIt is told in the Gospels that the Pharisees (sp.?) and scribes bribed\nthe Roman soldiers to say that the Diciples stole his body in the night.\nGood enough excuse for the Jewish and Roman objectives (of that day).\n\n--Clator\n--lbutler@hubcap.clemson.edu\n",15,soc.religion.christian
3099,"From: kennyc@cbnewsk.cb.att.com (kenneth.r.crudup)\nSubject: Re: V4 V6 V8 V12 Vx?\nOrganization: AT&T\nLines: 20\n\nIn article <1993Apr21.191744.3072@ole.cdac.com>\nssave@ole.cdac.com (The Devil Reincarnate) writes:\n\n>I am curious about knowing which commericial cars today\n>have V engines.\n\n>V8 - Don't know of any.\n\nKidding, right? \n\nCorvette, several MBZ's and BMW's, Mustang GT, etc., etc. There's a lot of\nthem. You from a European site?\n\n\t-Kenny\n\n-- \nKenny Crudup, ATT BL, MV20-3-T-5-B, X3219.\nkenny@mvuts.att.com\n\n\n",7,rec.autos
16847,"From: Mike.Hahn@p57.f714.n7102.z5.fidonet.org (Mike Hahn)\nSubject: The doctrine of Original Sin\nLines: 32\n\nStephen A. Creps writes to All:\n\n[...]\n\n SAC> Also, we know that\n SAC> the Bible says that _everyone_ must be baptized to enter Heaven.\n\nWhere exactly does it say that?\n\n SAC> _Everyone_ includes infants, unless there is other Scripture to the\n SAC> contrary, i.e. an exception. Since there is no exception listed in the\n SAC> Bible, we must assume (to be on the safe side) that the Bible means what\n SAC> it says, that _everyone_ must be baptized to enter Heaven.\n\nI think we do see an exception in the case of Cornelius and his\nhousehold, mentioned in Acts. Of course, they were baptised, but only\nafter ""God showed that He accepted them by giving them the Holy\nSpirit"". This means they were already acceptable to God before their\nbaptism, and had they suddenly died they would have gone to heaven.\n\nIn case that seems far-fetched - an ancestor of mine was a missionary\nwho worked among the Hereros in Namibia. Some of the tribesmen were\njealous of Christianity, and they poisoned the first convert before he\ncould be baptised. Surely he still went to heaven? I'm inclined to\nagree with a comment recorded at the time: ""It is not the neglect of\nbaptism, but its contempt, that condemns.""\n\nMike\n-- \nINTERNET: Mike.Hahn@p57.f714.n7102.z5.fidonet.org\nvia: THE CATALYST BBS in Port Elizabeth, South Africa.\n (catpe.alt.za) +27-41-34-2859, V32bis & HST.\n",15,soc.religion.christian


### Preprocess data

In [ ]:
# Define stop words from Gensim
stop_words = set(STOPWORDS)

# Initialize lemmatizer
lemmatizer = WordNetLemmatizer()

# Cache get_wordnet_pos for efficiency
@lru_cache(maxsize=128)
def get_wordnet_pos(treebank_tag):
    """Convert POS tag to WordNet POS tag using a dictionary lookup."""
    tag_dict = {
        'J': wordnet.ADJ,
        'V': wordnet.VERB,
        'N': wordnet.NOUN,
        'R': wordnet.ADV
    }
    return tag_dict.get(treebank_tag[0], wordnet.NOUN)  # Default to NOUN

In [ ]:
def clean_text(text):
    """Tokenize, remove stopwords, tag POS, and lemmatize efficiently."""
    tokens = [word for word in simple_preprocess(text, deacc=True, min_len=3) if word not in stop_words]
    return [lemmatizer.lemmatize(word, get_wordnet_pos(tag)) for word, tag in pos_tag(tokens)]

In [ ]:
%%time

df['cleaned_text'] = df['text'].apply(clean_text)

CPU times: user 2min 40s, sys: 2.35 s, total: 2min 42s
Wall time: 2min 50s


In [ ]:
df[['text','cleaned_text']].sample(3)

text  \
1109                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    

In [ ]:
%%time

  # Detect phrases
  phrases = Phrases(df['cleaned_text'],
                    min_count=5,     # Minimum # of times a phrase must appear
                    threshold=100    # Threshold for forming a phrase
                    )
  bigram = Phraser(phrases)
  df['phrased_text'] = df['cleaned_text'].apply(lambda tokens: bigram[tokens])

CPU times: user 13.4 s, sys: 1.01 s, total: 14.5 s
Wall time: 24.6 s


In [ ]:
df[['text','cleaned_text','phrased_text']].sample(3)

,text,cleaned_text,phrased_text
5871,"From: jmeritt@mental.MITRE.ORG (Jim Meritt - System Admin)\nSubject: Booze it up, thus sayth the Lord!\nOrganization: UTexas Mail-to-News Gateway\nLines: 8\nNNTP-Posting-Host: cs.utexas.edu\n\nJeremiah:\n25:27 Therefore thou shalt say unto them, Thus saith the LORD of\nhosts, the God of Israel; Drink ye, and be drunken, and spue, and\nfall, and rise no more, because of the sword which I will send among\nyou.\n25:28 And it shall be, if they refuse to take the cup at thine hand to\ndrink, then shalt thou say unto them, Thus saith the LORD of hosts; Ye\nshall certainly drink.\n","[jmeritt, mental, mitre, org, jim, meritt, admin, subject, booze, sayth, lord, organization, utexas, mail, news, gateway, line, nntp, post, host, utexas, edu, jeremiah, thou, shalt, unto, saith, lord, host, god, israel, drink, drunken, spue, fall, rise, sword, send, shall, refuse, cup, thine, hand, drink, shalt, thou, unto, saith, lord, host, shall, certainly, drink]","[jmeritt_mental, mitre_org, jim_meritt, admin, subject, booze, sayth, lord, organization, utexas, mail, news, gateway, line, nntp_post, host, utexas, edu, jeremiah, thou_shalt, unto, saith_lord, host, god, israel, drink, drunken, spue, fall, rise, sword, send, shall, refuse, cup, thine, hand, drink, shalt, thou, unto, saith_lord, host, shall, certainly, drink]"
2530,"From: mfeldman@bu.edu (Michael Feldman)\nSubject: Floptical Kills Superdrive\nLines: 13\nX-Newsreader: Tin 1.1 PL5\n\nI recently bought a PLI 21mgbyte floptical drive, and I was very happy \nwith it until I tried to use it to format a 1.4 HD diskette. I put the\nHD floppy in my Superdrive to check that the floptical had formatted it\ncorrectly, and now my Superdrive refuses to recognize ANY floppy (it says\n""this disk is unreadable"" and asks if I want to format it) even original\nsystems floppies from Apple. Nor will it format the disks if I try to\n(""initialization failed!"") Strangely enough the floptical still reads\nboth the 21 MB and 1.4 HD disks, but I cant look at my 800k floppies, and\nif I have a crash I'm screwed because the Floptical can't be used as a\nstart-up disk. PLI has been unresponsive. Any ideas? Has this happened\nto anyone before? I was looking for an inexpensive storage solution, and\nnow I am looking at an expensive repair. Help! respond to this thread, or\nemail mfeldman@acs.bu.edu\n","[mfeldman, edu, michael, feldman, subject, floptical, kill, superdrive, line, newsreader, tin, recently, buy, pli, mgbyte, floptical, drive, happy, try, use, format, diskette, floppy, superdrive, check, floptical, format, correctly, superdrive, refuse, recognize, floppy, say, disk, unreadable, asks, want, format, original, system, floppy, apple, format, disk, try, initialization, fail, strangely, floptical, read, disk, look, floppy, crash, screw, floptical, start, disk, pli, unresponsive, idea, happen, look, inexpensive, storage, solution, look, expensive, repair, help, respond, thread, email, mfeldman, acs, edu]","[mfeldman, edu, michael, feldman, subject, floptical, kill, superdrive, line, newsreader_tin, recently, buy, pli, mgbyte, floptical, drive, happy, try, use, format, diskette, floppy, superdrive, check, floptical, format, correctly, superdrive, refuse, recognize, floppy, say, disk, unreadable, asks, want, format, original, system, floppy, apple, format, disk, try, initialization, fail, strangely, floptical, read, disk, look, floppy, crash, screw, floptical, start, disk, pli, unresponsive, idea, happen, look, inexpensive, storage, solution, look, expensive, repair, help, respond, thread, email, mfeldman, acs, edu]"
10515,"From: kotsines@ucsu.Colorado.EDU (T. Kotsines)\nSubject: Re: SCSI vs. IDE\nNntp-Posting-Host: ucsu.colorado.edu\nOrganization: University of Colorado, Boulder\nLines: 17\n\nIn article <IISAKKIL.93Apr23125341@beta.hut.fi> iisakkil@beta.hut.fi (Mika Iisakkila) writes:\n>randy@msc.cornell.edu writes:\n>>Do all SCSI cards for DOS systems require a separate d

### Train Gensim Word2Vec model

In [ ]:
%%time

# Train Word2Vec model
w2v_model = Word2Vec(sentences=df['phrased_text'],
                     vector_size=100,  # Number of dimensions for word vectors (embedding size)
                     window=5,         # Maximum distance between target and context words
                     min_count=2,      # Ignores words that appear fewer than min_count times
                     epochs=10,        # Number of training iterations
                     sg=1,             # 1=Skip-gram model, 0=CBOW model
                     workers=4)

CPU times: user 3min 58s, sys: 3.38 s, total: 4min 2s
Wall time: 1min 37s


In [ ]:
# Create function to convert query into a vector for searching
def get_document_vector(tokens, model):
    """Get document vector by averaging word vectors."""
    vectors = [model.wv[word] for word in tokens if word in model.wv]
    if vectors:
        return np.mean(vectors, axis=0)
    else:
        return np.zeros(model.vector_size)

In [ ]:
# Convert to vectors for searching
df['document_vector'] = df['phrased_text'].apply(lambda tokens: get_document_vector(tokens, w2v_model))

In [ ]:
# Create function to make recommendation
def get_recommendations(doc_index, df, model, topn=5):
    query_vector = df['document_vector'][doc_index]
    similarities = df['document_vector'].apply(lambda vec: np.dot(vec, query_vector) / (np.linalg.norm(vec) * np.linalg.norm(query_vector)))
    similar_docs = similarities.sort_values(ascending=False).index[1:topn + 1]
    return df.loc[similar_docs][['text', 'target_names']]

In [ ]:
# Example: Get recommendations for a document
doc_index = 1
recommendations = get_recommendations(doc_index, df, w2v_model)

print(f"Search Document (Index {doc_index}):\n{df['text'][doc_index]}\n")
print("Recommendations:\n")
for index, row in recommendations.iterrows():
    print(f"Document (Index {index}):\n{row['text']}\nCategory: {row['target_names']}\n{'-'*80}\n")

Search Document (Index 1):
From: mblawson@midway.ecn.uoknor.edu (Matthew B Lawson)
Subject: Which high-performance VLB video card?
Summary: Seek recommendations for VLB video card
Nntp-Posting-Host: midway.ecn.uoknor.edu
Organization: Engineering Computer Network, University of Oklahoma, Norman, OK, USA
Keywords: orchid, stealth, vlb
Lines: 21

  My brother is in the market for a high-performance video card that supports
VESA local bus with 1-2MB RAM.  Does anyone have suggestions/ideas on:

  - Diamond Stealth Pro Local Bus

  - Orchid Farenheit 1280

  - ATI Graphics Ultra Pro

  - Any other high-performance VLB card


Please post or email.  Thank you!

  - Matt

-- 
    |  Matthew B. Lawson <------------> (mblawson@essex.ecn.uoknor.edu)  |   
  --+-- "Now I, Nebuchadnezzar, praise and exalt and glorify the King  --+-- 
    |   of heaven, because everything he does is right and all his ways  |   
    |   are just." - Nebuchadnezzar, king of Babylon, 562 B.C.           |   


Recommen

In [ ]:
# Example: Get recommendations for a differemt document
doc_index = 15
recommendations = get_recommendations(doc_index, df, w2v_model)

print(f"Search Document (Index {doc_index}):\n{df['text'][doc_index]}\n")
print("Recommendations:\n")
for index, row in recommendations.iterrows():
    print(f"Document (Index {index}):\n{row['text']}\nCategory: {row['target_names']}\n{'-'*80}\n")

Search Document (Index 15):
From: dbd@urartu.sdpa.org (David Davidian)
Subject: "Stretching from the Adriatic Sea to the Great Wall of China"
Organization: S.D.P.A. Center for Regional Studies
Lines: 22

In the following report: _Turkey Eyes Regional Role_ ANKARA, Turkey (AP)
April 27, 1993, we find in the last paragraph:

[Turanist] Although Premier Suleyman Demirel criticized Ozal's often
[Turanist] brash calls for more Turkish influence, he also has spoken
[Turanist] of a swath of Turkic peoples "stretching from the Adriatic
[Turanist] Sea to the Great Wall of China."

Who does Demirel think he is fooling? It seems at both ends of his envisioned 
pan-Turkic Empire -- the Balkans and the Caucasus -- Turkey's fascist boasts
are being pre-empted.

I would suggest Turkey let the world feel some of their "Grey Wolf Teeth", and
attempt to stretch from the Adriatic to China! Turkey will have cried "wolf"
just once too much! 


-- 
David Davidian dbd@urartu.sdpa.org   | "Armenia has not lea